# Solutions · 01 · Time response

Worked solutions to the exercises of [notebook 01](../notebooks/01_time_response.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, response as rs, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

For the DC motor, find the voltage step response's rise time and settling time and relate them to the two time constants. Then double the inertia J and repeat: which time constant moved?

In [2]:
for J in (0.01, 0.02):
    m = models.dc_motor(J=J)
    G = m["system"]
    t, y = rs.step(G, 3.0, 3001)
    info = rs.step_info(t, y, final=G.dcgain()[0, 0])
    print(f"J = {J}: time constants {np.round(-1 / G.poles().real, 4)} s; rise time {info['rise_time']:.3f} s, settling {info['settling_time']:.3f} s")

J = 0.01: time constants [0.4994 0.1   ] s; rise time 1.135 s, settling 2.066 s
J = 0.02: time constants [0.4992 0.2001] s; rise time 1.231 s, settling 2.209 s


The mechanical time constant (about J/b corrected by the back-emf) doubles with the inertia and the response
slows in proportion; the electrical one (L/R) is untouched. The rise and settling times follow the slow pole,
as the dominant-pole rule says.

## Exercise 2

A specification asks for overshoot < 10 % and 2 % settling < 1 s. Draw the allowed region of the s-plane (zeta and sigma bounds) and check it with three pole pairs of your choice by simulation.

In [3]:
zeta_min = -np.log(0.10) / np.sqrt(np.pi**2 + np.log(0.10) ** 2)      # overshoot < 10 %
sigma_min = 4.0 / 1.0                                                   # 2 % settling < 1 s (approximation)
print(f"allowed region: zeta >= {zeta_min:.3f} (angle from the negative real axis <= {np.degrees(np.arccos(zeta_min)):.1f} deg) and Re(s) <= -{sigma_min}")
for p in (-5 + 5j, -4 + 7j, -2 + 2j):
    zeta, wn = -p.real / abs(p), abs(p)
    G = sy.tf([wn**2], [1.0, 2 * zeta * wn, wn**2])
    t, y = rs.step(G, 4.0, 4001)
    info = rs.step_info(t, y, final=1.0)
    ok = info["overshoot"] < 10 and info["settling_time"] < 1.0
    print(f"poles {p}: zeta {zeta:.3f}, sigma {-p.real}: overshoot {info['overshoot']:.1f} %, settling {info['settling_time']:.2f} s -> {'meets' if ok else 'fails'} the specification")

allowed region: zeta >= 0.591 (angle from the negative real axis <= 53.8 deg) and Re(s) <= -4.0
poles (-5+5j): zeta 0.707, sigma 5.0: overshoot 4.3 %, settling 0.84 s -> meets the specification
poles (-4+7j): zeta 0.496, sigma 4.0: overshoot 16.6 %, settling 1.01 s -> fails the specification
poles (-2+2j): zeta 0.707, sigma 2.0: overshoot 4.3 %, settling 2.11 s -> fails the specification


The specification is a wedge of the s-plane: left of σ = 4 and inside the lines at ±54° from the negative real
axis. The pair at −5 ± 5j lies inside and meets both; −4 ± 7j has too little damping (overshoot 17 %); −2 ± 2j is
well damped but too slow. The settling-time rule is approximate, so a pole pair near σ = 4 should be checked
by simulation, as here.

## Exercise 3

**Implement it yourself:** compute the impulse response of the prototype from the initial-condition response with x0 = B (no input) and compare with `response.impulse`; explain why they are the same.

In [4]:
G = sy.tf([4.0], [1.0, 1.2, 4.0])
S = G.to_ss()
t, y_init, _ = rs.initial(S, S.B[:, 0], 10.0, 1001)       # no input, x0 = B
t, y_imp = rs.impulse(G, 10.0, 1001)
print("initial-condition response from x0 = B vs impulse response:", f"{np.max(np.abs(y_init - y_imp)):.1e}")

initial-condition response from x0 = B vs impulse response: 0.0e+00


An impulse at t = 0 transfers the state instantly from 0 to B (the integral of B δ(t)); afterwards the system
evolves freely. The impulse response is therefore the initial-condition response from x₀ = B, plus D δ(t) if
there is a direct term - exactly how the library computes it.